# Framework validation: protocol, independent repeats and manuscript exports

This is the canonical synthetic validation entrypoint. The first three notebooks inspect individual generating controls; this notebook executes a frozen experiment matrix with independent seeds and explicit failure denominators.
Smoke checks software. Development uses a larger sensitivity matrix with engineering estimator settings. The manuscript profile proposes the larger production-configuration study; completing a profile does not automatically validate a scientific claim.
The current protocol uses conventional c-GC/c-GC* and fastC-GC/fastC-GC* as four related CSL methods. Native lag support has no VAR coefficient readout. Histories are at least 256 samples so the fast complete-shift test can attain beta=0.005. This revised protocol supersedes the earlier eight-method pilot and writes to its own output directory.

Environment: select the project Python 3.14 `.venv` kernel. Install the locked packages with `uv sync --python 3.14 --locked --extra dev --extra csl --extra community --extra notebooks` from the repository root. The setup cell checks installed package versions; no upstream repository clone is required. Stored outputs may predate the backend migration; rerun the notebook to produce results with the current backend.


In [ ]:
import logging
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    _REPOSITORY_ROOT = next(
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "notebooks" / "_shared.py").is_file()
    )
except StopIteration as error:
    raise RuntimeError(
        "Could not locate the effectome repository from the notebook working directory"
    ) from error
sys.path[:0] = [
    path
    for path in (str(_REPOSITORY_ROOT / "src"), str(_REPOSITORY_ROOT))
    if path not in sys.path
]

if sys.version_info[:2] != (3, 14):
    raise RuntimeError(
        "Select the project Python 3.14 .venv kernel before running this notebook. "
        f"Current interpreter: {sys.executable}"
    )
_BACKEND_DISTRIBUTIONS = ("fastcgc", "tigramite", "scikit-learn", "kmedoids", "igraph", "leidenalg")
try:
    _BACKEND_VERSIONS = {name: version(name) for name in _BACKEND_DISTRIBUTIONS}
except PackageNotFoundError as error:
    raise RuntimeError(
        "The notebook kernel is missing a required installed backend. Run "
        "uv sync --python 3.14 --locked --extra dev --extra csl --extra community --extra notebooks "
        "from the repository root, then restart the project .venv kernel."
    ) from error
print("Notebook interpreter:", sys.executable)
print("Installed backends:", _BACKEND_VERSIONS)

ROOT = _REPOSITORY_ROOT
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logging.getLogger("effectome").setLevel(logging.INFO)
_CONVENTIONAL_PYTHON = ROOT / "tools/conventional-cgc/.venv/bin/python"
print("Conventional CPU runtime:", _CONVENTIONAL_PYTHON)
print("Runtime installed:", _CONVENTIONAL_PYTHON.is_file())


In [ ]:
import json

from IPython.display import Markdown, display

from effectome.experiments.framework_validation import (
    framework_validation_cells,
    load_framework_validation_profile,
    run_framework_validation,
)
from effectome.experiments.framework_validation_analysis import analyze_framework_validation
from effectome.workflows.cross_method import load_cross_method_profile
from notebooks._shared import PROJECT_ROOT

PROFILE = "smoke"  # smoke | development | manuscript
RUN_VALIDATION = False  # Explicitly enable after inspecting the declared matrix.
RENDER_VISUALS = False  # Explicitly regenerate the manuscript visual bundle.
MAX_CELLS = None  # A cap produces a partial report and cannot complete the publication suite.
validation_config = load_framework_validation_profile(PROFILE)
validation_cells = framework_validation_cells(validation_config)
_, validation_methods = load_cross_method_profile(validation_config.estimator_profile)
OUTPUT_DIRECTORY = (
    PROJECT_ROOT / "outputs" / "framework-validation"
    / validation_config.estimator_profile / PROFILE
)
print("Independent seeds:", validation_config.seeds)
print("Planned raw-process cells:", len(validation_cells))
print("Planned method runs:", len(validation_methods) * len(validation_cells))
print("Methods:", tuple(validation_methods))
print("Estimation profile:", validation_config.estimator_profile)
print("Outputs:", OUTPUT_DIRECTORY)


## Execute or inspect the fixed validation matrix

Positive log-loss gain means graph augmentation improves on the same activity/history baseline. All methods use the same held-out event origins. Histories, dwell times, horizons and observation stresses are independent declared factors. A stationary no-switching case tests estimation-induced graph-code multiplicity; this quantity is not a calibrated false-discovery rate.


In [ ]:
validation_report = None
if RUN_VALIDATION:
    validation_report = run_framework_validation(
        validation_config, OUTPUT_DIRECTORY, max_cells=MAX_CELLS,
    )
elif (OUTPUT_DIRECTORY / "validation.json").is_file():
    validation_report = json.loads((OUTPUT_DIRECTORY / "validation.json").read_text())
else:
    print("No results exist. Inspect the profile, set RUN_VALIDATION=True and run this cell.")

if validation_report is not None:
    validation_statistics = analyze_framework_validation(
        validation_report, OUTPUT_DIRECTORY / "analysis",
    )
    print("Method runs available/attempted:", validation_report["available_methods"],
          validation_report["attempted_methods"])
    print("Publication matrix complete:", validation_report["publication_suite_complete"])
    display(Markdown((OUTPUT_DIRECTORY / "analysis/analysis-report.md").read_text()))
    display(Markdown((OUTPUT_DIRECTORY / "analysis/numeric-summary.md").read_text()))


## Manuscript-ready artifacts and interpretation gates

The analysis directory contains exact numeric tables, seed-bootstrap 95% intervals, exact sign tests with full planned-family Holm correction, vector figures, a statistical appendix and generated LaTeX. Small seed counts and unavailable runs remain visible. The manuscript protocol must distinguish the executed bounded experiment from the proposed larger study.
Predicted transition probabilities and omission scores validate predictive operations, not biological necessity. The matched-marginal control concerns equal snapshot distributions; it does not imply equal history distributions. Independent-animal replication and intervention experiments remain separate gates.


In [ ]:
if validation_report is not None:
    from IPython.display import Image

    for filename in ("figure-01-predictive-gain.png", "figure-02-availability.png"):
        display(Image(filename=str(OUTPUT_DIRECTORY / "analysis/figures" / filename)))
    print("LaTeX table:", OUTPUT_DIRECTORY / "analysis/summary.tex")
    print("Executed-run text:", OUTPUT_DIRECTORY / "analysis/manuscript_results.tex")
    print("Statistical details:", OUTPUT_DIRECTORY / "analysis/stats-appendix.md")


## See the controls, transition predictions and neuron/group contributions

The visual bundle makes three different questions visible. Figure 03 shows the generating circuits, raw two-neuron occupancy and lagged dependence for each designed control. Its activity panels are raw projections, not an inferred manifold. Figure 04 shows a shared held-out manifold example beside observed and method-predicted transition matrices. Figure 05 shows transition-specific neuron/group omission scores, including negative scores.
The worked comparison uses the first declared matched-marginal specification and seed 0; it is not selected for a favorable result. State codes are local to this one fitted example. Do not pool numeric state labels across seeds. The numeric tables retain the repeated-study means, sample standard deviations, seed counts and unavailable results.
The same `visuals/` directory also contains compact manuscript versions of Figure 01 (four predictive-gain forest panels) and Figure 02 (available/attempted counts). Together the bundle provides five figures and two tables. The earlier cells display the original analysis versions of Figures 01–02.
Set `RENDER_VISUALS=True` to regenerate these presentation artifacts from the frozen validation results. This does not execute the full validation matrix. Existing visuals are displayed when present. The visual provenance distinguishes the frozen experiment from its presentation code.


In [ ]:
VISUAL_DIRECTORY = OUTPUT_DIRECTORY / "visuals"
if RENDER_VISUALS:
    from synthetic.validation_figures import render_validation_visuals

    visual_manifest = render_validation_visuals(OUTPUT_DIRECTORY, VISUAL_DIRECTORY)
    print("Visual manifest:", visual_manifest)

if (VISUAL_DIRECTORY / "visual-provenance.json").is_file():
    from IPython.display import Image

    for filename in (
        "figure-03-designed-controls.png",
        "figure-04-shared-transitions.png",
        "figure-05-transition-attribution.png",
    ):
        path = VISUAL_DIRECTORY / filename
        if path.is_file():
            display(Image(filename=str(path)))
    for filename in ("table-01-predictive-summary.md", "table-02-control-design.md"):
        path = VISUAL_DIRECTORY / filename
        if path.is_file():
            display(Markdown(path.read_text()))
    display(Markdown((VISUAL_DIRECTORY / "figure-catalog.md").read_text()))
    print("Visual provenance:", VISUAL_DIRECTORY / "visual-provenance.json")
else:
    print("No visual bundle exists. After validation, set RENDER_VISUALS=True and run this cell.")
